In [1]:
!pip install -q "transformers>=4.51.0" datasets torch accelerate peft sentencepiece protobuf


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import os
import random
import time
import gc
from datetime import datetime, timedelta
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup, Adafactor
from peft import LoraConfig, get_peft_model, TaskType

# ============================================================
# CONFIGURATION
# ============================================================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME   = "Qwen/Qwen3-1.7B"
MAX_LENGTH   = 1024
MAX_NEW_TOKENS = 2048

# Training hyperparameters
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
LORA_RANK        = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.1
NUM_TRAIN        = 1000
NUM_TEST         = 100
NUM_EPOCHS       = 2
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM    = 1.0
WARMUP_RATIO     = 0.1

# ---- Fine-tuning mode ----
USE_LORA = True   # Set to False for full fine-tuning (no PEFT)


# Concept-vector construction
NUM_CONCEPT_SAMPLES = 50
CONCEPT_MAX_ATTEMPTS = 100
DAILYDIALOG_CONFIG = "full"
CONCEPT_LAYER = -1

# Composite loss weights (following LLM-JEPA defaults)
GAMMA   = 1.0   # CE loss weight
LAMBDA_ = 0.1   # Cosine similarity (JEPA) loss weight
LAST_TOKEN = -3   # Qwen offset to skip <|im_end|> and trailing tokens


# JEPA predictor tokens (following LLM-JEPA architecture)
NUM_PREDICTORS = 1   

# Language
LANG_CONFIG = "pa"   # Punjabi native script
LANG_NAME   = "Punjabi"

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

PROMPT_TEMPLATE = (
    "Solve this math problem. Show your work step by step "
    "and give the final numeric answer.\n\nQuestion: {question}"
)

# Derived
steps_per_epoch = NUM_TRAIN // GRAD_ACCUM_STEPS
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print("=" * 70)
print("  INDIC-JEPA: Composite Loss SFT on Qwen3-1.7B")
print("=" * 70)
print(f"  Model:                {MODEL_NAME}")
print(f"  Language:             {LANG_NAME} ({LANG_CONFIG})")
print(f"  Train / Test:         {NUM_TRAIN} / {NUM_TEST}")
print(f"  Epochs:               {NUM_EPOCHS}")
print(f"  Grad accumulation:    {GRAD_ACCUM_STEPS}")
print(f"  Effective batch size: {GRAD_ACCUM_STEPS}")
print(f"  Steps per epoch:      {steps_per_epoch}")
print(f"  Total optim. steps:   {total_steps}")
print(f"  Warmup steps:         {warmup_steps}")
print(f"  LR:                   {LEARNING_RATE}")
print(f"  Loss:                 {GAMMA}·CE + {LAMBDA_}·(1 − CosSim)")
print(f"  Predictor tokens:     {NUM_PREDICTORS} (after PA question)")
print(f"  Concept samples:      {NUM_CONCEPT_SAMPLES} (GSM8K + DailyDialog)")
if USE_LORA:
    print(f"  LoRA:                 rank={LORA_RANK} α={LORA_ALPHA} drop={LORA_DROPOUT}")
else:
    print(f"  Fine-tuning:          FULL (all parameters, no LoRA)")
print(f"  Forward passes/sample:2 (PA full CE + PA Q+pred, static target)")
print("=" * 70)



In [15]:
# ============================================================
# LOAD MODEL + (optional LoRA) + PREDICTOR TOKENS
# ============================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Add predictor special tokens (following LLM-JEPA)
pred_tokens = [f"<|predictor_{i}|>" for i in range(1, NUM_PREDICTORS + 1)]
new_tokens = [t for t in pred_tokens if t not in tokenizer.get_vocab()]
if new_tokens:
    tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
    print(f"Added {len(new_tokens)} predictor tokens: {new_tokens}")

PRED_TOKEN_IDS = [tokenizer.convert_tokens_to_ids(t) for t in pred_tokens]
print(f"Predictor token IDs: {PRED_TOKEN_IDS}")

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False,
)

# Resize embeddings for new predictor tokens
if new_tokens:
    model.resize_token_embeddings(len(tokenizer))
    print(f"Resized embeddings → {len(tokenizer)} tokens")

if USE_LORA:
    print("Applying LoRA adapters...")
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_config)
    model.enable_input_require_grads()
else:
    print("Full fine-tuning mode (no LoRA)")
    # For full FT, all params already require grad by default.
    # enable_input_require_grads is only needed for PEFT.

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Print trainable param count (works for both modes)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")

device = next(model.parameters()).device
print(f"Device: {device}")
print(f"Dtype:  {next(model.parameters()).dtype}")


In [16]:
# ============================================================
# LOAD PAIRED DATASET (sarvamai/gsm8k-indic, Punjabi config)
# Each row contains BOTH the Punjabi question AND the original
# English question, so we get perfectly aligned pairs for free.
# ============================================================
print(f"Loading sarvamai/gsm8k-indic  config={LANG_CONFIG}...")
indic_ds = load_dataset("sarvamai/gsm8k-indic", LANG_CONFIG, split="test")
print(f"Total samples: {len(indic_ds)}")
print(f"Fields: {indic_ds.column_names}")

# Shuffle deterministically and split
all_indices = list(range(len(indic_ds)))
random.Random(SEED).shuffle(all_indices)

train_indices = all_indices[:NUM_TRAIN]
test_indices  = all_indices[NUM_TRAIN : NUM_TRAIN + NUM_TEST]
assert len(set(train_indices) & set(test_indices)) == 0, "Leak!"

train_samples = []
for idx in train_indices:
    r = indic_ds[idx]
    train_samples.append({
        "pa_question": r["question"],
        "pa_answer":   r["answer"],
        "en_question":  r["original_question"],
        "en_answer":    r["original_answer"],
    })

test_samples = []
for idx in test_indices:
    r = indic_ds[idx]
    test_samples.append({
        "pa_question": r["question"],
        "pa_answer":   r["answer"],
        "en_question":  r["original_question"],
        "en_answer":    r["original_answer"],
    })

print(f"Train: {len(train_samples)} paired (PA↔EN) samples")
print(f"Test:  {len(test_samples)} samples (unseen)")

# --- Verify pairing ---
print("\n--- Pairing verification (first 3) ---")
for i in range(3):
    s = train_samples[i]
    en_num = re.search(r'####\s*(-?[\d,]+\.?\d*)', s["en_answer"])
    pa_num = re.search(r'####\s*(-?[\d,]+\.?\d*)', s["pa_answer"])
    en_n = en_num.group(1) if en_num else "?"
    pa_n = pa_num.group(1) if pa_num else "?"
    print(f"[{i}] EN: {s['en_question'][:60]}…  →  {en_n}")
    print(f"    PA: {s['pa_question'][:60]}…  →  {pa_n}")
    print(f"    Match: {en_n == pa_n}")


In [17]:
# ============================================================
# HELPERS
# ============================================================

def extract_gold_answer(answer_str):
    """Extract numeric answer from GSM8K '#### N' format."""
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', answer_str)
    return m.group(1).replace(",", "").strip() if m else None

def extract_model_answer(text):
    """Extract last numeric value from model output."""
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        n = re.sub(r'[^\d.\-]', '', boxed[-1])
        if n: return n
    for pat in [
        r'(?:answer|result|total)\s*(?:is|=|:)\s*(-?[\d,]+\.?\d*)',
        r'####\s*(-?[\d,]+\.?\d*)',
        r'=\s*(-?[\d,]+\.?\d*)\s*$',
    ]:
        ms = re.findall(pat, text, re.I | re.M)
        if ms: return ms[-1].replace(",", "").strip()
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    return nums[-1].replace(",", "").strip() if nums else None

def compare_answers(pred, gold):
    if pred is None or gold is None: return False
    try:    return abs(float(pred) - float(gold)) < 1e-3
    except: return pred.strip() == gold.strip()

def find_pred_token_pos(input_ids):
    """Find position of the LAST predictor token in the sequence."""
    pred_id = PRED_TOKEN_IDS[-1]
    positions = (input_ids[0] == pred_id).nonzero(as_tuple=True)[0]
    assert len(positions) > 0, f"Predictor token {pred_id} not found!"
    return positions[-1].item()

def find_en_hidden_pos(input_ids):
    """Find the hidden state extraction position for EN answer.
    Uses LAST_TOKEN offset from unpadded length, matching LLM-JEPA's
    _last_token_index for Qwen (offset=-3)."""
    seq_len = input_ids.shape[1]  # no padding in our setup
    return seq_len + LAST_TOKEN


def build_prompts(sample):
    """Return 4 prompts for one training sample.

    pa_full_text    : PA Q + PA answer       → CE loss (forward pass 1)
    pa_q_pred_text  : PA Q + [PRED] tokens   → h_pa    (forward pass 2)
    en_answer_text  : EN answer standalone    → h_en    (forward pass 3)
    pa_q_only_text  : PA Q only              → for label masking length
    """
    pa_q = PROMPT_TEMPLATE.format(question=sample["pa_question"])

    # 1) PA full: question + answer (for CE loss)
    pa_full_text = tokenizer.apply_chat_template(
        [{"role": "user",      "content": pa_q},
         {"role": "assistant", "content": sample["pa_answer"]}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    # 2) PA question + predictor tokens (source for JEPA)
    #    Pre-truncate question to guarantee predictor token survives truncation
    pa_q_with_pred = pa_q
    for k in range(NUM_PREDICTORS, 0, -1):
        pa_q_with_pred += f"<|predictor_{k}|>"

    # Check if the full sequence would exceed MAX_LENGTH
    _test_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": pa_q_with_pred}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )
    _test_ids = tokenizer.encode(_test_text)

    if len(_test_ids) > MAX_LENGTH:
        # Compute overhead: template tokens + predictor tokens
        _bare = tokenizer.apply_chat_template(
            [{"role": "user", "content": ""}],
            tokenize=False, add_generation_prompt=False, enable_thinking=False,
        )
        overhead = len(tokenizer.encode(_bare)) + NUM_PREDICTORS
        # Truncate the question content to fit within MAX_LENGTH
        q_ids = tokenizer.encode(pa_q, add_special_tokens=False)
        max_q_len = MAX_LENGTH - overhead
        q_ids = q_ids[:max_q_len]
        pa_q_trunc = tokenizer.decode(q_ids, skip_special_tokens=False)
        pa_q_with_pred = pa_q_trunc
        for k in range(NUM_PREDICTORS, 0, -1):
            pa_q_with_pred += f"<|predictor_{k}|>"

    pa_q_pred_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": pa_q_with_pred}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    # 3) EN answer standalone (target for JEPA)
    #    Encoded as an assistant message, matching LLM-JEPA's
    #    get_assistant_messages → apply_chat_template
    en_answer_text = tokenizer.apply_chat_template(
        [{"role": "assistant", "content": sample["en_answer"]}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    # 4) PA question only (for label mask length)
    pa_q_only_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": pa_q}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

    return pa_full_text, pa_q_pred_text, en_answer_text, pa_q_only_text


# ---- Verification ----
s0 = train_samples[0]
_pa_full, _pa_qp, _en_ans, _pa_q = build_prompts(s0)

_pa_full_ids = tokenizer.encode(_pa_full)
_pa_qp_ids   = tokenizer.encode(_pa_qp)
_en_ans_ids  = tokenizer.encode(_en_ans)
_pa_q_ids    = tokenizer.encode(_pa_q)

# Verify question prefix for label masking
assert _pa_full_ids[:len(_pa_q_ids)] == _pa_q_ids, "PA question prefix mismatch!"

# Verify pred token is in the question+pred sequence
_pa_qp_enc = tokenizer(_pa_qp, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
pred_pos = find_pred_token_pos(_pa_qp_enc["input_ids"])

# Verify EN answer encoding
_en_ans_enc = tokenizer(_en_ans, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)


# Verify EN answer extraction position (using LAST_TOKEN offset)
h_en_idx = find_en_hidden_pos(_en_ans_enc["input_ids"])

print("✓ Prompt structure verified.")
print(f"  PA Q-only tokens       : {len(_pa_q_ids)}")
print(f"  PA full (Q+A) tokens   : {len(_pa_full_ids)}")
print(f"  PA Q+pred tokens       : {len(_pa_qp_ids)}  (pred at position {pred_pos})")
print(f"  EN answer-only tokens  : {len(_en_ans_ids)}  (h_en at position {h_en_idx}, offset={LAST_TOKEN})")
print(f"  Answer tokens (PA)     : {len(_pa_full_ids) - len(_pa_q_ids)}")
print(f"\n  Token at pred pos ({pred_pos}): '{tokenizer.decode(_pa_qp_enc['input_ids'][0, pred_pos])}'")
print(f"  Token at h_en pos ({h_en_idx}): '{tokenizer.decode(_en_ans_enc['input_ids'][0, h_en_idx])}'")
print(f"\n  PA Q+pred decoded (last 30 chars): ...{_pa_qp[-30:]}")
print(f"  EN answer decoded (first 80 chars): {_en_ans[:80]}...")


In [18]:
# ============================================================
# CONCEPT VECTOR BUILD + BASELINE COSINE (pre-training)
# concept_vec = mean(GSM8K_correct) - mean(DailyDialog)
# ============================================================

def extract_last_hidden_state(text, model_obj):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    out = model_obj(**enc, output_hidden_states=True)
    h = out.hidden_states[CONCEPT_LAYER][0, -1, :].detach().float().cpu()
    del out
    return h


def format_english_gsm8k_sft(question, answer):
    q = PROMPT_TEMPLATE.format(question=question)
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}, {"role": "assistant", "content": answer}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )


def format_dailydialog_sft(question, answer):
    q = f"Respond naturally to this dialogue line:\n\n{question}"
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}, {"role": "assistant", "content": answer}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )


def generate_and_grade_gsm8k(question, gold_answer):
    prompt = PROMPT_TEMPLATE.format(question=question)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    grading_max_new_tokens = MAX_NEW_TOKENS

    with torch.no_grad():
        gen_ids = model.generate(
            **enc,
            max_new_tokens=grading_max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    continuation_ids = gen_ids[0, enc["input_ids"].shape[1]:]
    decoded = tokenizer.decode(continuation_ids, skip_special_tokens=True)
    pred = extract_model_answer(decoded)
    hit = compare_answers(pred, gold_answer)
    return hit, pred, decoded


def build_math_concept_vector():
    print("Building static concept vector...")
    print(f"  Target samples: {NUM_CONCEPT_SAMPLES} GSM8K-correct and {NUM_CONCEPT_SAMPLES} DailyDialog")

    gsm8k_en = load_dataset("openai/gsm8k", "main", split="train")

    def load_dailydialog_rows_via_server(max_rows=12000, page_size=50, max_retries=8, target_pairs=1000, pair_buffer=250):
        import urllib.parse
        import urllib.request
        import urllib.error

        rows = []
        offset = 0
        fetch_start = time.time()
        target_pair_budget = target_pairs + pair_buffer
        running_pair_estimate = 0

        while offset < max_rows:
            length = min(page_size, max_rows - offset)
            url = (
                "https://datasets-server.huggingface.co/rows?"
                + urllib.parse.urlencode({
                    "dataset": "roskoN/dailydialog",
                    "config": DAILYDIALOG_CONFIG,
                    "split": "train",
                    "offset": offset,
                    "length": length,
                })
            )

            payload = None
            for attempt in range(max_retries):
                try:
                    with urllib.request.urlopen(url, timeout=30) as resp:
                        payload = json.loads(resp.read().decode("utf-8"))
                    break
                except urllib.error.HTTPError as http_err:
                    if http_err.code != 429 or attempt == max_retries - 1:
                        raise
                    sleep_s = min(90, 2 ** attempt + random.random())
                    print(f"  DailyDialog API rate-limited (429). Retry {attempt+1}/{max_retries} after {sleep_s:.1f}s...")
                    time.sleep(sleep_s)
                except urllib.error.URLError:
                    if attempt == max_retries - 1:
                        raise
                    sleep_s = min(30, 2 ** attempt + random.random())
                    print(f"  DailyDialog API network issue. Retry {attempt+1}/{max_retries} after {sleep_s:.1f}s...")
                    time.sleep(sleep_s)

            page_rows = payload.get("rows", []) if payload is not None else []
            if not page_rows:
                break

            for item in page_rows:
                row = item.get("row", {})
                utterances = row.get("utterances")
                if utterances:
                    rows.append({"utterances": utterances})
                    running_pair_estimate += max(0, len(utterances) - 1)

            offset += len(page_rows)
            elapsed = time.time() - fetch_start
            rate = max(1e-6, offset / elapsed)
            eta_s = max(0, int((max_rows - offset) / rate))
            if offset % 500 == 0 or offset >= max_rows:
                print(
                    f"  DailyDialog rows fetched: {offset}/{max_rows} | "
                    f"pair_est~{running_pair_estimate}/{target_pair_budget} | "
                    f"{elapsed:.0f}s elapsed | ETA {eta_s}s"
                )

            if running_pair_estimate >= target_pair_budget:
                print(f"  DailyDialog fetch stop: collected estimated {running_pair_estimate} adjacent pairs.")
                break

            if len(page_rows) < length:
                break

        return rows

    try:
        dailydialog = load_dataset("roskoN/dailydialog", DAILYDIALOG_CONFIG, split="train")
        dailydialog_source = "roskoN/dailydialog"
        dailydialog_rows = dailydialog
    except RuntimeError as e:
        if "Dataset scripts are no longer supported" not in str(e):
            raise
        print("  roskoN/dailydialog uses dataset scripts in this environment; using dataset-server rows API.")
        dailydialog_rows = load_dailydialog_rows_via_server(
            max_rows=11118,
            target_pairs=NUM_CONCEPT_SAMPLES,
            pair_buffer=max(200, NUM_CONCEPT_SAMPLES // 4),
        )
        dailydialog_source = "roskoN/dailydialog (dataset-server rows API, early-stop)"

    gsm8k_indices = list(range(len(gsm8k_en)))
    random.shuffle(gsm8k_indices)

    gsm8k_states = []
    gsm8k_attempted = 0
    gsm8k_discarded = 0
    gsm8k_used_indices = []
    gsm_start = time.time()

    model.eval()
    with torch.no_grad():
        for idx in gsm8k_indices:
            if len(gsm8k_states) >= NUM_CONCEPT_SAMPLES:
                break
            if gsm8k_attempted >= CONCEPT_MAX_ATTEMPTS:
                break

            row = gsm8k_en[idx]
            q = row["question"]
            a = row["answer"]
            gold = extract_gold_answer(a)
            gsm8k_attempted += 1

            hit, _, _ = generate_and_grade_gsm8k(q, gold)
            if not hit:
                gsm8k_discarded += 1
            else:
                sft_text = format_english_gsm8k_sft(q, a)
                gsm8k_states.append(extract_last_hidden_state(sft_text, model))
                gsm8k_used_indices.append(idx)

            if gsm8k_attempted % 5 == 0 or len(gsm8k_states) == NUM_CONCEPT_SAMPLES:
                elapsed = time.time() - gsm_start
                attempt_rate = max(1e-6, gsm8k_attempted / elapsed)
                hit_rate = len(gsm8k_states) / max(1, gsm8k_attempted)
                if hit_rate > 0:
                    remaining = NUM_CONCEPT_SAMPLES - len(gsm8k_states)
                    eta_s = max(0, int(remaining / max(1e-6, attempt_rate * hit_rate)))
                else:
                    eta_s = -1
                eta_text = f"{eta_s}s" if eta_s >= 0 else "estimating..."
                print(
                    f"  GSM8K progress: correct {len(gsm8k_states)}/{NUM_CONCEPT_SAMPLES}, "
                    f"attempted {gsm8k_attempted}/{CONCEPT_MAX_ATTEMPTS}, "
                    f"discarded {gsm8k_discarded}, elapsed {elapsed:.0f}s, ETA {eta_text}"
                )

    if len(gsm8k_states) < NUM_CONCEPT_SAMPLES:
        raise ValueError(
            f"Only {len(gsm8k_states)} correct GSM8K samples found after {gsm8k_attempted} attempts. "
            "Increase CONCEPT_MAX_ATTEMPTS or reduce NUM_CONCEPT_SAMPLES."
        )

    dd_pairs = []
    for row_idx, row in enumerate(dailydialog_rows):
        utt = row["utterances"]
        for i in range(len(utt) - 1):
            q = utt[i].strip()
            a = utt[i + 1].strip()
            if q and a:
                dd_pairs.append((row_idx, i, q, a))

    if len(dd_pairs) < NUM_CONCEPT_SAMPLES:
        raise ValueError(f"DailyDialog has only {len(dd_pairs)} valid adjacent pairs.")

    random.shuffle(dd_pairs)
    dd_selected = dd_pairs[:NUM_CONCEPT_SAMPLES]
    dd_states = []
    dd_start = time.time()

    with torch.no_grad():
        for j, (_, _, q, a) in enumerate(dd_selected, start=1):
            sft_text = format_dailydialog_sft(q, a)
            dd_states.append(extract_last_hidden_state(sft_text, model))
            if j % 25 == 0 or j == NUM_CONCEPT_SAMPLES:
                elapsed = time.time() - dd_start
                rate = max(1e-6, j / elapsed)
                eta_s = max(0, int((NUM_CONCEPT_SAMPLES - j) / rate))
                print(f"  DailyDialog forward-pass progress: {j}/{NUM_CONCEPT_SAMPLES}, elapsed {elapsed:.0f}s, ETA {eta_s}s")

    gsm8k_tensor = torch.stack(gsm8k_states, dim=0)
    dailydialog_tensor = torch.stack(dd_states, dim=0)

    gsm8k_mean = gsm8k_tensor.mean(dim=0)
    dailydialog_mean = dailydialog_tensor.mean(dim=0)
    concept_vec_cpu = gsm8k_mean - dailydialog_mean
    concept_vec_norm_cpu = F.normalize(concept_vec_cpu, dim=0)

    concept_stats = {
        "num_concept_samples": NUM_CONCEPT_SAMPLES,
        "gsm8k_attempted": gsm8k_attempted,
        "gsm8k_correct": len(gsm8k_states),
        "gsm8k_discarded": gsm8k_discarded,
        "dailydialog_source": dailydialog_source,
        "dailydialog_used": len(dd_states),
        "dailydialog_total_pairs": len(dd_pairs),
        "concept_layer": CONCEPT_LAYER,
        "concept_norm": float(concept_vec_cpu.norm().item()),
        "gsm8k_used_indices": gsm8k_used_indices,
        "dailydialog_used_pairs": [(int(r), int(i)) for (r, i, _, _) in dd_selected],
    }

    return concept_vec_cpu, concept_vec_norm_cpu, concept_stats


concept_vec_cpu, concept_vec_norm_cpu, concept_stats = build_math_concept_vector()
concept_vec = concept_vec_cpu.to(device)
concept_vec_norm = concept_vec_norm_cpu.to(device)

print("\nConcept vector ready.")
print(f"  GSM8K attempted/correct/discarded: {concept_stats['gsm8k_attempted']} / {concept_stats['gsm8k_correct']} / {concept_stats['gsm8k_discarded']}")
print(f"  DailyDialog used pairs: {concept_stats['dailydialog_used']}")
print(f"  ||concept_vec||: {concept_stats['concept_norm']:.4f}")

# ---- Baseline cosine vs static concept vector ----
NUM_BASELINE = 50
print(f"\nComputing pre-training cosine similarities ({NUM_BASELINE} samples)...")
print("  Source: PA question + [PRED]  →  Target: static concept vector")
model.eval()
baseline_cos_sims = []

with torch.no_grad():
    for i in range(NUM_BASELINE):
        s = train_samples[i]
        _, pa_qp, _, _ = build_prompts(s)

        pa_enc = tokenizer(pa_qp, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        out_pa = model(**pa_enc, output_hidden_states=True)
        h_pa = out_pa.hidden_states[-1][0, find_pred_token_pos(pa_enc["input_ids"]), :].float()

        h_pa_norm = F.normalize(h_pa, dim=0)
        cs = torch.dot(h_pa_norm, concept_vec_norm.float()).item()
        baseline_cos_sims.append(cs)

baseline_mean = sum(baseline_cos_sims) / len(baseline_cos_sims)
baseline_std = (sum((x - baseline_mean) ** 2 for x in baseline_cos_sims) / len(baseline_cos_sims)) ** 0.5
print("\nPre-training PA_Q+[PRED] ↔ static concept-vector cosine similarity:")
print(f"  Mean: {baseline_mean:.4f}  (std {baseline_std:.4f})")
print(f"  Min:  {min(baseline_cos_sims):.4f}")
print(f"  Max:  {max(baseline_cos_sims):.4f}")


In [19]:
# ============================================================
# TRAINING LOOP — static concept-vector JEPA
#   Pass 1: PA Q + PA answer  → CE loss on answer tokens
#   Pass 2: PA Q + [PRED]     → h_pa at pred token
#   Static target: concept_vec = mean(GSM8K_correct)-mean(DailyDialog)
# ============================================================
print("=" * 70)
print("  STARTING TRAINING")
print("  Source: PA Q + [PRED]  →  Target: static concept vector")
print("=" * 70)

model.train()

optimizer = Adafactor(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    relative_step=False,      # We manage LR ourselves via scheduler
    scale_parameter=False,    # Disable auto-scaling since we set LR explicitly
    warmup_init=False,        # Disable built-in warmup since we use our own
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


training_log = []
global_step = 0
accumulated = 0
optimizer.zero_grad()
total_start = time.time()
sample_times = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_ce, epoch_jepa, epoch_cos, epoch_total = [], [], [], []

    epoch_samples = train_samples.copy()
    random.shuffle(epoch_samples)

    print(f"\n{'=' * 70}")
    print(f"  EPOCH {epoch + 1}/{NUM_EPOCHS}")
    print(f"{'=' * 70}")

    for i, sample in enumerate(epoch_samples):
        t0 = time.time()

        # ---- BUILD PROMPTS ----
        pa_full_text, pa_qp_text, _, pa_q_only = build_prompts(sample)
        pa_q_len = len(tokenizer.encode(pa_q_only))

        # ---- TOKENISE ----
        pa_full_enc = tokenizer(pa_full_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        pa_qp_enc = tokenizer(pa_qp_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

        # ---- LABELS (mask question, keep answer) ----
        labels = pa_full_enc["input_ids"].clone()
        mask_end = min(pa_q_len, labels.shape[1])
        labels[0, :mask_end] = -100

        # Forward 1: CE
        outputs_ce = model(
            input_ids=pa_full_enc["input_ids"],
            attention_mask=pa_full_enc["attention_mask"],
            labels=labels,
        )
        ce_loss = outputs_ce.loss

        # Forward 2: Punjabi predictor state
        outputs_pa = model(
            input_ids=pa_qp_enc["input_ids"],
            attention_mask=pa_qp_enc["attention_mask"],
            output_hidden_states=True,
        )
        h_pa_pos = find_pred_token_pos(pa_qp_enc["input_ids"])
        h_pa = outputs_pa.hidden_states[-1][0, h_pa_pos, :].float()

        # Static target JEPA loss
        h_pa_norm = F.normalize(h_pa, dim=0)
        cos_sim = torch.dot(h_pa_norm, concept_vec_norm.float())
        jepa_loss = 1.0 - cos_sim

        feat_penalty = 1e-4 * h_pa.pow(2).mean()
        
        total_loss = GAMMA * ce_loss + LAMBDA_ * jepa_loss

        (total_loss / GRAD_ACCUM_STEPS).backward()
        accumulated += 1

        if accumulated >= GRAD_ACCUM_STEPS:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            accumulated = 0
            global_step += 1

        dt = time.time() - t0
        sample_times.append(dt)
        cv, jv, cosv, tv = ce_loss.item(), jepa_loss.item(), cos_sim.item(), total_loss.item()
        epoch_ce.append(cv)
        epoch_jepa.append(jv)
        epoch_cos.append(cosv)
        epoch_total.append(tv)

        training_log.append({
            "epoch": epoch + 1,
            "sample": i + 1,
            "global_sample": epoch * NUM_TRAIN + i + 1,
            "global_step": global_step,
            "ce_loss": round(cv, 6),
            "jepa_loss": round(jv, 6),
            "cosine_sim": round(cosv, 6),
            "total_loss": round(tv, 6),
            "lr": scheduler.get_last_lr()[0],
            "step_time_s": round(dt, 3),
        })

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == NUM_TRAIN:
            window = min(50, len(epoch_ce))
            avg_t = sum(sample_times[-window:]) / window
            rem = (NUM_EPOCHS - epoch) * NUM_TRAIN - (i + 1)
            eta = str(timedelta(seconds=int(rem * avg_t)))
            print(
                f"  [{i+1:3d}/{NUM_TRAIN}] "
                f"CE:{sum(epoch_ce[-window:])/window:.4f}  "
                f"JEPA:{sum(epoch_jepa[-window:])/window:.4f}  "
                f"CosSim:{sum(epoch_cos[-window:])/window:.4f}  "
                f"Total:{sum(epoch_total[-window:])/window:.4f}  "
                f"LR:{scheduler.get_last_lr()[0]:.1e}  "
                f"{avg_t:.2f}s/samp  ETA:{eta}"
            )

        if (i + 1) % 100 == 0:
            step_ckpt_path = f"{RESULTS_DIR}/checkpoint_epoch_{epoch+1}_step_{i+1}"
            print(f"\n  Saving step checkpoint to {step_ckpt_path}...")
            model.save_pretrained(step_ckpt_path)
            tokenizer.save_pretrained(step_ckpt_path)

        del outputs_ce, outputs_pa, h_pa, cos_sim, jepa_loss, total_loss, ce_loss
        torch.cuda.empty_cache()

    if accumulated > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        accumulated = 0
        global_step += 1

    et = time.time() - epoch_start
    print(f"\n  ── Epoch {epoch+1} summary ──")
    print(f"     CE Loss   : {sum(epoch_ce)/len(epoch_ce):.4f}")
    print(f"     JEPA Loss : {sum(epoch_jepa)/len(epoch_jepa):.4f}")
    print(f"     CosSim    : {sum(epoch_cos)/len(epoch_cos):.4f}")
    print(f"     Total Loss: {sum(epoch_total)/len(epoch_total):.4f}")
    print(f"     Time      : {et:.0f}s  ({et/60:.1f} min)")
    print(f"     Opt. steps: {global_step}")

    epoch_ckpt_path = f"{RESULTS_DIR}/checkpoint_epoch_{epoch+1}"
    print(f"\n  Saving checkpoint to {epoch_ckpt_path}...")
    model.save_pretrained(epoch_ckpt_path)
    tokenizer.save_pretrained(epoch_ckpt_path)

tt = time.time() - total_start
print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE — {tt:.0f}s ({tt/60:.1f} min)")
print(f"  Total optimizer steps: {global_step}")
print(f"{'='*70}")


In [20]:
# ============================================================
# POST-TRAINING: cosine similarity comparison (vs concept vector)
# ============================================================
print("Computing post-training cosine similarities...")
model.eval()
post_cos_sims = []

with torch.no_grad():
    for i in range(NUM_BASELINE):
        s = train_samples[i]
        _, pa_qp, _, _ = build_prompts(s)

        pa_enc = tokenizer(pa_qp, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        out_pa = model(**pa_enc, output_hidden_states=True)
        h_pa = out_pa.hidden_states[-1][0, find_pred_token_pos(pa_enc["input_ids"]), :].float()

        cs = torch.dot(F.normalize(h_pa, dim=0), concept_vec_norm.float()).item()
        post_cos_sims.append(cs)

post_mean = sum(post_cos_sims) / len(post_cos_sims)
post_std = (sum((x - post_mean) ** 2 for x in post_cos_sims) / len(post_cos_sims)) ** 0.5

print(f"\n{'':18s} {'Pre-Train':>12s} {'Post-Train':>12s} {'Δ':>10s}")
print("-" * 56)
print(f"  {'Mean':12s}   {baseline_mean:10.4f}   {post_mean:10.4f}   {post_mean-baseline_mean:+8.4f}")
print(f"  {'Std':12s}   {baseline_std:10.4f}   {post_std:10.4f}")
print(f"  {'Min':12s}   {min(baseline_cos_sims):10.4f}   {min(post_cos_sims):10.4f}   {min(post_cos_sims)-min(baseline_cos_sims):+8.4f}")
print(f"  {'Max':12s}   {max(baseline_cos_sims):10.4f}   {max(post_cos_sims):10.4f}   {max(post_cos_sims)-max(baseline_cos_sims):+8.4f}")


In [21]:
# ============================================================
# EVALUATION — 100 unseen Punjabi samples
# ============================================================
print("=" * 70)
print("  EVALUATION: 100 unseen Punjabi GSM8K samples")
print("=" * 70)

model.gradient_checkpointing_disable()   # not needed for inference
model.eval()

eval_logs  = []
correct    = 0
eval_start = time.time()

for i, sample in enumerate(test_samples):
    question  = sample["pa_question"]
    gold_raw  = sample["pa_answer"]
    gold      = extract_gold_answer(gold_raw)
    if gold is None:
        gold = extract_gold_answer(sample["en_answer"])

    prompt  = PROMPT_TEMPLATE.format(question=question)
    msgs    = [{"role": "user", "content": prompt}]
    text    = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, top_p=0.8, top_k=20,
            do_sample=True,
        )
    out_ids  = gen[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    pred     = extract_model_answer(response)
    hit      = compare_answers(pred, gold)
    if hit:
        correct += 1

    eval_logs.append({
        "index": i, "pa_question": question,
        "en_question": sample["en_question"],
        "gold": gold, "pred": pred, "correct": hit,
        "response_preview": response[:400],
    })

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:3d}/{NUM_TEST}]  "
              f"Gold:{str(gold):>6s}  Pred:{str(pred):>6s}  "
              f"{'✓' if hit else '✗'}   "
              f"Running: {correct}/{i+1} = {correct/(i+1)*100:.1f}%")

eval_time = time.time() - eval_start
accuracy  = correct / NUM_TEST * 100

print(f"\n{'='*70}")
print(f"  RESULT: {correct}/{NUM_TEST} = {accuracy:.1f}%")
print(f"  Time:   {eval_time:.0f}s  ({eval_time/NUM_TEST:.1f}s / sample)")
print(f"{'='*70}")


In [ ]:
# ============================================================
# SAVE EVERYTHING
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary = {
    "model": MODEL_NAME,
    "method": "Indic-JEPA with static math concept vector",
    "language": f"{LANG_NAME} ({LANG_CONFIG})",
    "timestamp": timestamp,
    "config": {
        "seed": SEED, "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "grad_accum": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "fine_tuning_mode": "LoRA" if USE_LORA else "Full",
        "lora_rank": LORA_RANK if USE_LORA else None,
        "lora_alpha": LORA_ALPHA if USE_LORA else None,
        "gamma_ce": GAMMA, "lambda_jepa": LAMBDA_,
        "num_train": NUM_TRAIN, "num_test": NUM_TEST,
        "forward_passes_per_sample": 2,
    },

    "concept_vector": concept_stats,
    "cosine_similarity": {
        "pre_training": {"mean": round(baseline_mean, 4), "std": round(baseline_std, 4)},
        "post_training": {"mean": round(post_mean, 4), "std": round(post_std, 4)},
        "delta_mean": round(post_mean - baseline_mean, 4),
    },
    "eval": {
        "correct": correct, "total": NUM_TEST,
        "accuracy": round(accuracy, 2),
        "time_s": round(eval_time, 1),
    },
    "training_time_s": round(tt, 1),
}

all_data = {
    "summary": summary,
    "training_log": training_log,
    "eval_logs": eval_logs,
    "baseline_cos_sims": [round(x, 6) for x in baseline_cos_sims],
    "post_cos_sims":     [round(x, 6) for x in post_cos_sims],
}

log_path = f"{RESULTS_DIR}/indic_jepa_train_{timestamp}.json"
sum_path = f"{RESULTS_DIR}/indic_jepa_summary_{timestamp}.json"

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Logs saved    → {log_path}")
print(f"Summary saved → {sum_path}")

# ---- Final report ----
print(f"\n{'='*70}")
print(f"  FINAL REPORT — Indic-JEPA on {MODEL_NAME}")
print(f"{'='*70}")
print(f"  Language:          {LANG_NAME}")
print(f"  Training samples:  {NUM_TRAIN} × {NUM_EPOCHS} epochs")
print(f"  Forward passes:    2 (PA full CE + PA Q+pred, static target)")
print(f"  Training time:     {tt/60:.1f} min")
print(f"")
print(f"  Cosine Similarity (PA predictor ↔ concept vector):")
print(f"    Pre-training:    {baseline_mean:.4f}")
print(f"    Post-training:   {post_mean:.4f}  (Δ = {post_mean-baseline_mean:+.4f})")
print(f"")
print(f"  Evaluation (100 unseen PA samples):")
print(f"    Accuracy:        {correct}/{NUM_TEST} = {accuracy:.1f}%")
print(f"{'='*70}")

# Save model
if USE_LORA:
    adapter_path = f"{RESULTS_DIR}/lora_adapter_{timestamp}"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"\nLoRA adapter saved → {adapter_path}/")
else:
    ft_path = f"{RESULTS_DIR}/full_ft_model_{timestamp}"
    model.save_pretrained(ft_path)
    tokenizer.save_pretrained(ft_path)
    print(f"\nFull fine-tuned model saved → {ft_path}/")



In [ ]:
# ============================================================
# BASELINE SFT: Load a FRESH model (no JEPA-trained weights)
# ============================================================
import gc

# Free JEPA model memory
del model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()

print("Loading fresh model for baseline SFT...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False,
)

if USE_LORA:
    lora_config_bl = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    base_model = get_peft_model(base_model, lora_config_bl)
    base_model.enable_input_require_grads()
else:
    print("Full fine-tuning mode (no LoRA)")

base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in base_model.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")
print("✓ Fresh model loaded for baseline SFT")



In [ ]:
# ============================================================
# BASELINE TRAINING LOOP — CE ONLY, 1 forward pass / sample
# Same hyperparams, same data, same schedule
# ============================================================
print("=" * 70)
print("  STARTING BASELINE SFT (CE only, no JEPA)")
print("=" * 70)

BASELINE_RESULTS_DIR = "results_baseline_sft"
os.makedirs(BASELINE_RESULTS_DIR, exist_ok=True)

base_model.train()

bl_optimizer = Adafactor(
    filter(lambda p: p.requires_grad, base_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    relative_step=False,
    scale_parameter=False,
    warmup_init=False,
)

bl_scheduler = get_linear_schedule_with_warmup(
    bl_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

bl_training_log = []
bl_global_step  = 0
bl_accumulated  = 0
bl_optimizer.zero_grad()
bl_total_start  = time.time()
bl_sample_times = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_ce = []

    epoch_samples = train_samples.copy()
    random.shuffle(epoch_samples)

    print(f"\n{'='*70}")
    print(f"  EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    for i, sample in enumerate(epoch_samples):
        t0 = time.time()

        # ---- BUILD PROMPT (reuse existing helper) ----
        pa_full_text, pa_q_text, _ = build_prompts(sample)
        pa_q_len = len(tokenizer.encode(pa_q_text))

        # ---- TOKENISE ----
        pa_enc = tokenizer(pa_full_text, return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)

        # ---- LABELS (mask question, keep answer — same as JEPA) ----
        labels = pa_enc["input_ids"].clone()
        mask_end = min(pa_q_len, labels.shape[1])
        labels[0, :mask_end] = -100

        # =========================================================
        # SINGLE FORWARD PASS — CE loss only (no English, no JEPA)
        # =========================================================
        outputs = base_model(
            input_ids=pa_enc["input_ids"],
            attention_mask=pa_enc["attention_mask"],
            labels=labels,
        )
        ce_loss = outputs.loss

        # ---- BACKWARD ----
        (ce_loss / GRAD_ACCUM_STEPS).backward()
        bl_accumulated += 1

        if bl_accumulated >= GRAD_ACCUM_STEPS:
            torch.nn.utils.clip_grad_norm_(base_model.parameters(), MAX_GRAD_NORM)
            bl_optimizer.step()
            bl_scheduler.step()
            bl_optimizer.zero_grad()
            bl_accumulated = 0
            bl_global_step += 1

        # ---- RECORD ----
        dt = time.time() - t0
        bl_sample_times.append(dt)
        cv = ce_loss.item()
        epoch_ce.append(cv)

        bl_training_log.append({
            "epoch": epoch + 1, "sample": i + 1,
            "global_sample": epoch * NUM_TRAIN + i + 1,
            "global_step": bl_global_step,
            "ce_loss": round(cv, 6),
            "lr": bl_scheduler.get_last_lr()[0],
            "step_time_s": round(dt, 3),
        })

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == NUM_TRAIN:
            window = min(50, len(epoch_ce))
            avg_t = sum(bl_sample_times[-window:]) / window
            rem   = (NUM_EPOCHS - epoch) * NUM_TRAIN - (i + 1)
            eta   = str(timedelta(seconds=int(rem * avg_t)))
            print(
                f"  [{i+1:3d}/{NUM_TRAIN}] "
                f"CE:{sum(epoch_ce[-window:])/window:.4f}  "
                f"LR:{bl_scheduler.get_last_lr()[0]:.1e}  "
                f"{avg_t:.2f}s/samp  ETA:{eta}"
            )

        # ---- SAVE CHECKPOINT (EVERY 100 SAMPLES) ----
        if (i + 1) % 100 == 0:
            step_ckpt_path = f"{BASELINE_RESULTS_DIR}/checkpoint_epoch_{epoch+1}_step_{i+1}"
            print(f"\n  Saving step checkpoint to {step_ckpt_path}...")
            base_model.save_pretrained(step_ckpt_path)
            tokenizer.save_pretrained(step_ckpt_path)
            
        del outputs, ce_loss
        torch.cuda.empty_cache()

    # Flush remaining
    if bl_accumulated > 0:
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), MAX_GRAD_NORM)
        bl_optimizer.step(); bl_scheduler.step(); bl_optimizer.zero_grad()
        bl_accumulated = 0; bl_global_step += 1

    et = time.time() - epoch_start
    print(f"\n  ── Epoch {epoch+1} summary ──")
    print(f"     CE Loss   : {sum(epoch_ce)/len(epoch_ce):.4f}")
    print(f"     Time      : {et:.0f}s  ({et/60:.1f} min)")
    print(f"     Opt. steps: {bl_global_step}")

    # ---- SAVE CHECKPOINT (END OF EPOCH) ----
    epoch_ckpt_path = f"{BASELINE_RESULTS_DIR}/checkpoint_epoch_{epoch+1}"
    print(f"\n  Saving epoch checkpoint to {epoch_ckpt_path}...")
    base_model.save_pretrained(epoch_ckpt_path)
    tokenizer.save_pretrained(epoch_ckpt_path)

bl_tt = time.time() - bl_total_start
print(f"\n{'='*70}")
print(f"  BASELINE TRAINING COMPLETE — {bl_tt:.0f}s ({bl_tt/60:.1f} min)")
print(f"  Total optimizer steps: {bl_global_step}")
print(f"{'='*70}")


In [ ]:
# ============================================================
# BASELINE EVALUATION — same 100 unseen samples
# ============================================================
print("=" * 70)
print("  BASELINE EVALUATION: 100 unseen Punjabi GSM8K samples")
print("=" * 70)

base_model.gradient_checkpointing_disable()
base_model.eval()

bl_eval_logs = []
bl_correct   = 0
bl_eval_start = time.time()

for i, sample in enumerate(test_samples):
    question = sample["pa_question"]
    gold_raw = sample["pa_answer"]
    gold     = extract_gold_answer(gold_raw)
    if gold is None:
        gold = extract_gold_answer(sample["en_answer"])

    prompt = PROMPT_TEMPLATE.format(question=question)
    msgs   = [{"role": "user", "content": prompt}]
    text   = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen = base_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, top_p=0.8, top_k=20,
            do_sample=True,
        )
    out_ids  = gen[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    pred     = extract_model_answer(response)
    hit      = compare_answers(pred, gold)
    if hit:
        bl_correct += 1

    bl_eval_logs.append({
        "index": i, "pa_question": question,
        "en_question": sample["en_question"],
        "gold": gold, "pred": pred, "correct": hit,
        "response_preview": response[:400],
    })

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:3d}/{NUM_TEST}]  "
              f"Gold:{str(gold):>6s}  Pred:{str(pred):>6s}  "
              f"{'✓' if hit else '✗'}   "
              f"Running: {bl_correct}/{i+1} = {bl_correct/(i+1)*100:.1f}%")

bl_eval_time = time.time() - bl_eval_start
bl_accuracy  = bl_correct / NUM_TEST * 100

print(f"\n{'='*70}")
print(f"  BASELINE RESULT: {bl_correct}/{NUM_TEST} = {bl_accuracy:.1f}%")
print(f"  Time: {bl_eval_time:.0f}s  ({bl_eval_time/NUM_TEST:.1f}s / sample)")
print(f"{'='*70}")


In [ ]:
# ============================================================
# SAVE BASELINE & COMPARE WITH JEPA
# ============================================================
bl_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

bl_summary = {
    "model": MODEL_NAME,
    "method": "Baseline SFT (CE only, no JEPA)",
    "language": f"{LANG_NAME} ({LANG_CONFIG})",
    "timestamp": bl_timestamp,
    "config": {
        "seed": SEED, "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "grad_accum": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "loss": "CE only",
        "num_train": NUM_TRAIN, "num_test": NUM_TEST,
        "forward_passes_per_sample": 1,
    },
    "eval": {
        "correct": bl_correct, "total": NUM_TEST,
        "accuracy": round(bl_accuracy, 2),
        "time_s": round(bl_eval_time, 1),
    },
    "training_time_s": round(bl_tt, 1),
}

bl_all_data = {
    "summary": bl_summary,
    "training_log": bl_training_log,
    "eval_logs": bl_eval_logs,
}

bl_log_path = f"{BASELINE_RESULTS_DIR}/baseline_sft_train_{bl_timestamp}.json"
bl_sum_path = f"{BASELINE_RESULTS_DIR}/baseline_sft_summary_{bl_timestamp}.json"

with open(bl_log_path, "w", encoding="utf-8") as f:
    json.dump(bl_all_data, f, ensure_ascii=False, indent=2)
with open(bl_sum_path, "w", encoding="utf-8") as f:
    json.dump(bl_summary, f, ensure_ascii=False, indent=2)

# Save baseline LoRA adapter
bl_adapter_path = f"{BASELINE_RESULTS_DIR}/lora_adapter_{bl_timestamp}"
base_model.save_pretrained(bl_adapter_path)
tokenizer.save_pretrained(bl_adapter_path)

print(f"Baseline logs    → {bl_log_path}")
print(f"Baseline summary → {bl_sum_path}")
print(f"Baseline adapter → {bl_adapter_path}/")

# ============================================================
# HEAD-TO-HEAD COMPARISON
# ============================================================
# Pull JEPA results from earlier cells (already in runtime)
jepa_accuracy = accuracy   # from Cell 9
jepa_time     = tt         # from Cell 7

print(f"\n{'='*70}")
print(f"  HEAD-TO-HEAD: JEPA vs Baseline SFT")
print(f"{'='*70}")
print(f"  {'':30s} {'JEPA SFT':>12s} {'Baseline':>12s} {'Δ':>8s}")
print(f"  {'-'*64}")
print(f"  {'Eval Accuracy (%)':30s} {jepa_accuracy:>11.1f}% {bl_accuracy:>11.1f}% {bl_accuracy - jepa_accuracy:>+7.1f}%")
print(f"  {'Training Time (min)':30s} {jepa_time/60:>11.1f}  {bl_tt/60:>11.1f}  {bl_tt/60 - jepa_time/60:>+7.1f}")
print(f"  {'Fwd Passes / Sample':30s} {'2':>12s} {'1':>12s}")
print(f"  {'Loss':30s} {'CE+CosSim':>12s} {'CE only':>12s}")
print(f"  {'CosSim Δ (post−pre)':30s} {post_mean - baseline_mean:>+11.4f}  {'N/A':>12s}")
print(f"{'='*70}")
